# Tools and Tool Descriptions

**What you will build:** an agent with several tools, and an understanding of the one thing that most controls whether it uses them well — the tool **description**. This maps to chapter 07 of the n8n course.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/12_tools_and_descriptions.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once per notebook.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.5 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"   # any id from openrouter.ai/models
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## Two ways to register a tool

- `@agent.tool_plain` — for a tool that needs nothing from the agent's context (most tools).
- `@agent.tool` — for a tool that needs the run context (`ctx`), e.g. to reach injected dependencies (that's notebook 1.3).

The function's **type hints** become the argument schema, and its **docstring** becomes the description the model reads. You write normal Python; PydanticAI turns it into a tool.

In [ ]:
agent = Agent(model, system_prompt="You are a helpful assistant. Use tools when they help.")

@agent.tool_plain
def get_stock_price(ticker: str) -> str:
    """Get the latest price for a stock ticker symbol, e.g. 'AAPL'."""
    fake = {"AAPL": 224.10, "GOOGL": 178.30, "TSLA": 251.44}
    return f"{ticker}: ${fake.get(ticker.upper(), 100.0)}"

@agent.tool_plain
def convert_currency(amount: float, to_currency: str) -> str:
    """Convert an amount in US dollars to another currency (EUR, GBP, JPY)."""
    rates = {"EUR": 0.92, "GBP": 0.79, "JPY": 156.0}
    return f"{amount * rates.get(to_currency.upper(), 1.0):.2f} {to_currency.upper()}"

result = await agent.run("How much is 10 shares of AAPL worth in euros?")
print(result.output)

The agent chained two tools — get the price, then convert — with no routing code from you. It read the descriptions, decided the order, and combined the results.

### See what the decorator actually built

In 0.3 you wrote the JSON schema for every tool by hand. Claiming "the type hints generate it" deserves proof. PydanticAI ships a fake model for testing, `TestModel`, that records exactly what *would* be sent to a real model — including the generated tool definitions. (No tokens, no network; `TestModel` gets a whole chapter in 1.5b.)

In [ ]:
import json
from pydantic_ai.models.test import TestModel

probe = TestModel()
await agent.run("hi", model=probe)     # run-level model override; nothing real is called

for t in probe.last_model_request_parameters.function_tools:
    print(t.name)
    print("  description:", t.description)
    print("  schema:     ", json.dumps(t.parameters_json_schema))
    print()

Compare with the schemas you hand-wrote in 0.3, field by field: `type: object`, `properties` from the type hints, `required` from the non-default arguments, the description from the docstring. Nothing was invented — the decorator wrote your 0.3 boilerplate for you. And note the debugging power: whenever you wonder "what is the model actually being told about my tools?", this probe answers it in four lines.

## The description is the steering wheel

A tool's docstring is not documentation for humans — it's the instruction the model uses to decide **whether and when** to call the tool. Vague descriptions are the number-one cause of agents that ignore a tool or misuse it. Anthropic's research ([Writing effective tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents)) puts tool descriptions at the top of what determines agent quality. Let's prove it.

In [ ]:
# A tool with a DELIBERATELY vague description:
vague = Agent(model, system_prompt="You are a helpful assistant.")

@vague.tool_plain
def lookup(x: str) -> str:
    """Does a thing with input."""          # <- useless description
    fake = {"AAPL": 224.10, "GOOGL": 178.30}
    return f"{x}: ${fake.get(x.upper(), 100.0)}"


def tools_called(r):
    """Which tools did a run actually call? (Reusable — keep it around.)"""
    return [p.tool_name for m in r.all_messages()
            for p in m.parts if p.part_kind == "tool-call"]

# One run proves nothing (0.0: non-determinism). Run it FIVE times and count:
used = 0
for i in range(5):
    r = await vague.run("What is the price of GOOGL?")
    calls = tools_called(r)
    used += bool(calls)
    print(f"run {i+1}: called {calls or 'nothing — answered from its imagination'}")
print(f"\nvague description -> used the tool in {used}/5 runs")

In [ ]:
# Same function, same model, same question — only the DOCSTRING changes:
clear = Agent(model, system_prompt="You are a helpful assistant.")

@clear.tool_plain
def lookup(x: str) -> str:
    """Get the latest price for a stock ticker symbol, e.g. 'AAPL'.
    Use it whenever the user asks about a stock's current price."""
    fake = {"AAPL": 224.10, "GOOGL": 178.30}
    return f"{x}: ${fake.get(x.upper(), 100.0)}"

used = 0
for i in range(5):
    r = await clear.run("What is the price of GOOGL?")
    used += bool(tools_called(r))
print(f"clear description -> used the tool in {used}/5 runs")

Typical result: the vague version calls the tool inconsistently (or never — the model can't tell it's about stock prices, so it guesses); the clear version approaches 5/5. Two lessons in one experiment. First, **the description is the steering wheel** — you tune agent behavior far more through docstrings and the system prompt than through code. Second, notice *how* we had to check: five runs and a count, not one run and an impression. Evidence about agents is statistical; 1.6 turns that instinct into proper evals.

## When the *argument* is wrong: `ModelRetry`

Descriptions steer tool *choice*; the next failure mode is tool *arguments*. Ask for "the price of Apple" and the model may pass `"Apple"` where the API needs `"AAPL"`. You could return an error string (0.3's approach) — but PydanticAI has a dedicated signal: raise **`ModelRetry`** inside the tool, and the framework sends your message back to the model as a correction and lets it try again. It's the retry-on-validation loop from 0.2, applied to tool arguments — and it's the canonical PydanticAI pattern for argument validation.

In [ ]:
from pydantic_ai import ModelRetry

strict_agent = Agent(model, system_prompt="You are a helpful assistant. Use tools when they help.")

@strict_agent.tool_plain
def stock_price(ticker: str) -> str:
    """Get the latest price for a stock ticker SYMBOL, e.g. 'AAPL'."""
    if not (ticker.isalpha() and ticker.isupper() and 1 <= len(ticker) <= 5):
        raise ModelRetry(
            f"{ticker!r} is not a ticker symbol. Pass the symbol in capital letters — "
            f"e.g. 'AAPL' for Apple, 'GOOGL' for Google."
        )
    fake = {"AAPL": 224.10, "GOOGL": 178.30}
    return f"{ticker}: ${fake.get(ticker, 100.0)}"

r = await strict_agent.run("What's the current price of Apple stock?")
print(r.output)
print("\ncalls made:", tools_called(r))

If the model's first call passed `"Apple"`, you'll see `stock_price` in the call list **twice** — the failed attempt and the corrected one. The `ModelRetry` message is doing the teaching: it names the problem *and* shows the fix, which is what makes the self-correction converge (the same principle as the judge feedback in 0.5). Guardrails (1.5) build on exactly this mechanism.

## A real tool: live weather

Every tool so far returned fake data, on purpose — it keeps cells deterministic and free, a legitimate teaching choice. But real tools bring real failure modes: latency, timeouts, rate limits, malformed responses. Here's a genuinely live tool using [wttr.in](https://wttr.in), a free keyless weather API. Note the `try/except` — a real tool *will* fail sometimes, and the agent should get a useful error instead of crashing (the lesson from 0.3).

In [ ]:
import httpx   # already installed by pydantic-ai; no extra dependency

weather_agent = Agent(model, system_prompt="You are a helpful assistant. Use tools when they help.")

@weather_agent.tool_plain
def live_weather(city: str) -> str:
    """Get the CURRENT real weather for a city, e.g. 'Bilbao'."""
    try:
        r = httpx.get(f"https://wttr.in/{city}?format=j1", timeout=10)
        cur = r.json()["current_condition"][0]
        return f"{cur['temp_C']}C, {cur['weatherDesc'][0]['value']}"
    except Exception as e:
        return f"Weather service unavailable: {e}"

result = await weather_agent.run("What's the weather like in Bilbao right now?")
print(result.output)

That was a real network round-trip — the model asked for `live_weather('Bilbao')`, your code called the API, and the result flowed back into the loop. The moment a tool touches the outside world, error handling stops being optional. Wrap it, set a timeout, and decide what the agent should see when it fails.

One refinement for later: `httpx.get` is *synchronous* — it briefly blocks the event loop, harmless in a notebook, undesirable in a server. PydanticAI supports async tools directly: declare `async def live_weather(...)` and use `httpx.AsyncClient`. Same tool, non-blocking; you'll want that form in Block 3.

::::{dropdown} 🔍 What makes a good tool description
:color: info

```
Bad:   "Does a thing with input."
Good:  "Get the latest price for a stock ticker symbol, e.g. 'AAPL'. Use this whenever
        the user asks about a stock's current or recent price."
```

A good description says: **what** it does, **what the arguments mean** (with an example), and **when** to use it. Argument names and type hints matter too — `ticker: str` reads better to the model than `x: str`.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a units trap.** Add a `temperature(city: str)` tool that returns Celsius, and ask in Fahrenheit. Measure it (5 runs, like the experiment above): how often does the model convert correctly? Then say the unit explicitly in the docstring and re-measure. **Done when:** you have two numbers, and the second is better.
2. **Name matters.** Rename `convert_currency`'s `to_currency` argument to `c` and re-run the AAPL-in-euros question a few times. **Done when:** you've seen (or measured) the reliability drop — argument names are part of the interface the model reads.
3. **Your own `ModelRetry`.** Add validation to `convert_currency`: if `to_currency` isn't one of EUR/GBP/JPY, raise `ModelRetry` listing the supported currencies. **Done when:** asking to convert "to pesos" produces either a corrected call or an honest "not supported" answer — not a silent wrong number.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Units trap:**

```python
@agent.tool_plain
def temperature(city: str) -> str:
    """Current temperature for a city, in CELSIUS. If the user wants Fahrenheit,
    convert: F = C * 9/5 + 32."""
    return "22"        # fake — the point is the unit, not the value
```

Before the docstring says the unit, the model has to *guess* what "22" means; afterwards it converts. A tool result is context, and unlabeled context invites hallucinated interpretation.

**2 — Name matters:** with `c: str` the model must infer the meaning from the description alone; expect more wrong or missing arguments. Descriptive names + type hints are free reliability.

**3 — Your own ModelRetry:**

```python
RATES = {"EUR": 0.92, "GBP": 0.79, "JPY": 156.0}

@agent.tool_plain
def convert_currency(amount: float, to_currency: str) -> str:
    """Convert an amount in US dollars to another currency (EUR, GBP or JPY)."""
    cur = to_currency.upper()
    if cur not in RATES:
        raise ModelRetry(f"{to_currency!r} is not supported. Supported: EUR, GBP, JPY.")
    return f"{amount * RATES[cur]:.2f} {cur}"
```

The retry message *lists the valid options* — give the model what it needs to correct itself, not just a rejection.
::::

::::{dropdown} 🔧 The "Think" tool
:color: info

A trick worth knowing: give the agent a tool that does **nothing** except let it write down its reasoning.

```python
@agent.tool_plain
def think(thought: str) -> str:
    """Use this to reason step by step before acting. It just records your thought."""
    return "noted"
```

On hard, multi-step tasks the model calls `think` to plan before calling real tools, which measurably improves reliability — many coding agents ship exactly this. It costs a turn, so use it only where planning actually helps. (The n8n course teaches the same tool in its tool-calling chapter.)
::::

::::{dropdown} 🔧 Common issues (and fixes)
:color: secondary

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **The model ignores a tool** | Description too vague, or unclear argument names | Rewrite the docstring to say what/when; use descriptive names + type hints |
| **Wrong tool chosen** | Two tools overlap in purpose | Make descriptions clearly distinct; say when NOT to use each |
| **Bad arguments passed** | Ambiguous argument meaning | Add an example to the docstring (e.g. "ticker like 'AAPL'") |
| **A real tool hangs the run** | No timeout on the network call | Always pass `timeout=` and wrap in `try/except` (see `live_weather`) |
::::

**What's next:** in **1.3** we add **typed output** and **dependency injection** — giving tools access to real data (a database, the current user) safely.